In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/mind-recommender"

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
TRAIN_DIR = os.path.join(DATA_DIR, "MINDsmall_train")
DEV_DIR = os.path.join(DATA_DIR, "MINDsmall_dev")
GLOVE_DIR = os.path.join(DATA_DIR, "glove")

RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
NOTEBOOKS_DIR = os.path.join(PROJECT_ROOT, "notebooks")

print("Connected to project:")
print(PROJECT_ROOT)

Connected to project:
/content/drive/MyDrive/mind-recommender


## Step 1: Text Tokenization and Vocabulary Construction

News titles were converted to lowercase and tokenized using NLTK. A vocabulary was constructed from the training dataset only, applying a minimum frequency threshold of 2 to remove rare words and reduce noise.

Special tokens `<PAD>` and `<UNK>` were included to handle sequence padding and unseen words during later stages. The final vocabulary size was 20,774 words.

In [ ]:
import pandas as pd
import nltk
from collections import Counter

nltk.download('punkt')
nltk.download('punkt_tab', quiet=True)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
news_cols = ['news_id', 'category', 'subcategory', 'title',
             'abstract', 'url', 'title_entities', 'abstract_entities']

train_news = pd.read_csv(
    os.path.join(TRAIN_DIR, "news.tsv"),
    sep="\t",
    names=news_cols
)

print("Train news shape:", train_news.shape)
display(train_news[['news_id', 'title']].head())

Train news shape: (51282, 8)


,news_id,title
0,N55528,"The Brands Queen Elizabeth, Prince Charles, an..."
1,N19639,50 Worst Habits For Belly Fat
2,N61837,The Cost of Trump's Aid Freeze in the Trenches...
3,N53526,I Was An NBA Wife. Here's How It Affected My M...
4,N38324,"How to Get Rid of Skin Tags, According to a De..."


In [ ]:
class NewsTokenizer:
    def __init__(self, max_title_len=30, min_word_freq=2):
        self.max_title_len = max_title_len
        self.min_word_freq = min_word_freq
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}

    def tokenize(self, text):
        if pd.isna(text):
            return []
        text = str(text).lower()
        return nltk.word_tokenize(text)

    def build_vocab(self, titles):
        word_counts = Counter()

        for title in titles:
            tokens = self.tokenize(title)
            word_counts.update(tokens)

        for word, count in word_counts.items():
            if count >= self.min_word_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word

        print(f"Vocabulary size: {len(self.word2idx)}")

    def encode_title(self, title):
        tokens = self.tokenize(title)

        indices = [
            self.word2idx.get(token, self.word2idx['<UNK>'])
            for token in tokens
        ]

        # Pad or truncate
        if len(indices) < self.max_title_len:
            indices += [self.word2idx['<PAD>']] * (self.max_title_len - len(indices))
        else:
            indices = indices[:self.max_title_len]

        return indices

In [ ]:
tokenizer = NewsTokenizer(max_title_len=30, min_word_freq=2)
tokenizer.build_vocab(train_news['title'])

Vocabulary size: 20774


In [ ]:
sample_title = train_news.loc[0, 'title']

print("Original title:")
print(sample_title)

print("\nTokenized title:")
print(tokenizer.tokenize(sample_title))

print("\nFirst 20 vocabulary items:")
print(list(tokenizer.word2idx.items())[:20])

Original title:
The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By

Tokenized title:
['the', 'brands', 'queen', 'elizabeth', ',', 'prince', 'charles', ',', 'and', 'prince', 'philip', 'swear', 'by']

First 20 vocabulary items:
[('<PAD>', 0), ('<UNK>', 1), ('the', 2), ('brands', 3), ('queen', 4), ('elizabeth', 5), (',', 6), ('prince', 7), ('charles', 8), ('and', 9), ('philip', 10), ('swear', 11), ('by', 12), ('50', 13), ('worst', 14), ('habits', 15), ('for', 16), ('belly', 17), ('fat', 18), ('cost', 19)]


## Step 2: Word Embedding Matrix

Using the vocabulary constructed from the training titles, a GloVe-based embedding matrix was created with shape `(vocab_size, 300)`. For each word in the vocabulary, its corresponding 300-dimensional pre-trained GloVe vector was inserted into the matrix when available.

Out of 20,774 words in the vocabulary, 19,069 were successfully matched with GloVe embeddings (~91% coverage). Words not found in GloVe were initialized randomly. The `<PAD>` token was assigned a zero vector to support sequence padding.

The final embedding matrix has shape (20774, 300) and will be used to initialize the embedding layer of the neural network.

In [ ]:
import numpy as np

def load_glove(glove_path, word2idx, embed_dim=300):
    embedding_matrix = np.random.normal(
        loc=0.0,
        scale=0.1,
        size=(len(word2idx), embed_dim)
    ).astype("float32")

    # PAD token should be all zeros
    embedding_matrix[word2idx["<PAD>"]] = np.zeros(embed_dim, dtype="float32")

    found = 0

    with open(glove_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            word = parts[0]

            if word in word2idx:
                vector = np.array(parts[1:], dtype="float32")

                if vector.shape[0] == embed_dim:
                    embedding_matrix[word2idx[word]] = vector
                    found += 1

    print(f"Found {found}/{len(word2idx)} words in GloVe")
    print("Embedding matrix shape:", embedding_matrix.shape)

    return embedding_matrix

In [ ]:
GLOVE_PATH = os.path.join(GLOVE_DIR, "glove.6B.300d.txt")

embedding_matrix = load_glove(
    glove_path=GLOVE_PATH,
    word2idx=tokenizer.word2idx,
    embed_dim=300
)

Found 19069/20774 words in GloVe
Embedding matrix shape: (20774, 300)


In [ ]:
print("PAD index:", tokenizer.word2idx["<PAD>"])
print("UNK index:", tokenizer.word2idx["<UNK>"])

print("PAD vector (first 10 values):")
print(embedding_matrix[tokenizer.word2idx["<PAD>"]][:10])

print("UNK vector (first 10 values):")
print(embedding_matrix[tokenizer.word2idx["<UNK>"]][:10])

PAD index: 0
UNK index: 1
PAD vector (first 10 values):
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
UNK vector (first 10 values):
[-0.1230571  -0.1298413  -0.1047975   0.05215325 -0.01633872  0.12059631
  0.10596877  0.12139563  0.1902538  -0.07665125]


In [ ]:
sample_word = "news"

if sample_word in tokenizer.word2idx:
    print(f"Index of '{sample_word}':", tokenizer.word2idx[sample_word])
    print(f"Embedding for '{sample_word}' (first 10 values):")
    print(embedding_matrix[tokenizer.word2idx[sample_word]][:10])
else:
    print(f"'{sample_word}' not found in vocabulary")

Index of 'news': 222
Embedding for 'news' (first 10 values):
[-0.41681   0.19399   0.22356   0.18028   0.33988   0.1587   -0.21046
  0.42863  -0.041325 -1.8492  ]


In [ ]:
np.save(os.path.join(RESULTS_DIR, "embedding_matrix.npy"), embedding_matrix)

print("Saved embedding_matrix.npy to results folder")

Saved embedding_matrix.npy to results folder


## Step 3: News Encoding Preparation

Each news title was converted into a fixed-length sequence of word indices using the vocabulary built from the training set. Titles shorter than 30 tokens were padded with the `<PAD>` token, while longer titles were truncated to maintain a consistent sequence length.

This step transforms raw text into a uniform numerical format that can be directly consumed by a neural network. The encoded news dictionaries created for the train and dev sets will be used later to retrieve title representations for user histories and candidate articles. For example, the first title was tokenized, mapped to indices, and padded with zeros at the end until it reached the required length of 30.

In [ ]:
news_cols = ['news_id', 'category', 'subcategory', 'title',
             'abstract', 'url', 'title_entities', 'abstract_entities']

dev_news = pd.read_csv(
    os.path.join(DEV_DIR, "news.tsv"),
    sep="\t",
    names=news_cols
)

print("Dev news shape:", dev_news.shape)
display(dev_news[['news_id', 'title']].head())

Dev news shape: (42416, 8)


,news_id,title
0,N55528,"The Brands Queen Elizabeth, Prince Charles, an..."
1,N18955,Dispose of unwanted prescription drugs during ...
2,N61837,The Cost of Trump's Aid Freeze in the Trenches...
3,N53526,I Was An NBA Wife. Here's How It Affected My M...
4,N38324,"How to Get Rid of Skin Tags, According to a De..."


In [ ]:
train_news['title_encoded'] = train_news['title'].apply(tokenizer.encode_title)
dev_news['title_encoded'] = dev_news['title'].apply(tokenizer.encode_title)

print("Encoded train titles and dev titles successfully.")

Encoded train titles and dev titles successfully.


In [ ]:
display(train_news[['news_id', 'title', 'title_encoded']].head())

print("Example encoded title length:", len(train_news.loc[0, 'title_encoded']))
print("Max title length setting:", tokenizer.max_title_len)

,news_id,title,title_encoded
0,N55528,"The Brands Queen Elizabeth, Prince Charles, an...","[2, 3, 4, 5, 6, 7, 8, 6, 9, 7, 10, 11, 12, 0, ..."
1,N19639,50 Worst Habits For Belly Fat,"[13, 14, 15, 16, 17, 18, 0, 0, 0, 0, 0, 0, 0, ..."
2,N61837,The Cost of Trump's Aid Freeze in the Trenches...,"[2, 19, 20, 21, 22, 23, 24, 25, 2, 26, 20, 27,..."
3,N53526,I Was An NBA Wife. Here's How It Affected My M...,"[29, 30, 31, 32, 33, 34, 35, 22, 36, 37, 38, 3..."
4,N38324,"How to Get Rid of Skin Tags, According to a De...","[36, 42, 43, 44, 20, 45, 46, 6, 47, 42, 48, 49..."


Example encoded title length: 30
Max title length setting: 30


In [ ]:
train_news_encoded = dict(zip(train_news['news_id'], train_news['title_encoded']))
dev_news_encoded = dict(zip(dev_news['news_id'], dev_news['title_encoded']))

all_news_encoded = {**train_news_encoded, **dev_news_encoded}

print("Train encoded news count:", len(train_news_encoded))
print("Dev encoded news count:", len(dev_news_encoded))
print("Total encoded news count:", len(all_news_encoded))

Train encoded news count: 51282
Dev encoded news count: 42416
Total encoded news count: 65238


In [ ]:
sample_idx = 0

print("Original title:")
print(train_news.loc[sample_idx, 'title'])

print("\nTokenized title:")
print(tokenizer.tokenize(train_news.loc[sample_idx, 'title']))

print("\nEncoded title:")
print(train_news.loc[sample_idx, 'title_encoded'])

Original title:
The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By

Tokenized title:
['the', 'brands', 'queen', 'elizabeth', ',', 'prince', 'charles', ',', 'and', 'prince', 'philip', 'swear', 'by']

Encoded title:
[2, 3, 4, 5, 6, 7, 8, 6, 9, 7, 10, 11, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
train_news[['news_id', 'title_encoded']].to_pickle(
    os.path.join(RESULTS_DIR, "train_news_encoded.pkl")
)

dev_news[['news_id', 'title_encoded']].to_pickle(
    os.path.join(RESULTS_DIR, "dev_news_encoded.pkl")
)

print("Saved train_news_encoded.pkl and dev_news_encoded.pkl")

Saved train_news_encoded.pkl and dev_news_encoded.pkl


## Step 4: Behavior Parsing

In this step, each user behavior record was parsed into three components: the user's click history, the positively clicked articles within an impression, and the negatively skipped articles.

The history was stored as a list of previously clicked news IDs, while impressions were split into positive (`-1`) and negative (`-0`) samples. On average, users had approximately 32 previously clicked articles, with around 1–2 positive clicks and 35 negative candidates per impression. This structured representation prepares the data for the next step, where negative sampling will be applied to construct training examples for the recommendation model.

In [ ]:
beh_cols = ['impression_id', 'user_id', 'time', 'history', 'impressions']

train_behaviors = pd.read_csv(
    os.path.join(TRAIN_DIR, "behaviors.tsv"),
    sep="\t",
    names=beh_cols
)

dev_behaviors = pd.read_csv(
    os.path.join(DEV_DIR, "behaviors.tsv"),
    sep="\t",
    names=beh_cols
)

print("Train behaviors shape:", train_behaviors.shape)
print("Dev behaviors shape:", dev_behaviors.shape)
display(train_behaviors.head())

Train behaviors shape: (156965, 5)
Dev behaviors shape: (73152, 5)


,impression_id,user_id,time,history,impressions
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
2,3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...
3,4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0
4,5,U8125,11/12/2019 4:11:21 PM,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...


In [ ]:
def split_impressions(impression_str):
    if pd.isna(impression_str):
        return [], []

    impressions = str(impression_str).split()

    pos = [imp.split('-')[0] for imp in impressions if imp.endswith('-1')]
    neg = [imp.split('-')[0] for imp in impressions if imp.endswith('-0')]

    return pos, neg

In [ ]:
def parse_behavior_row(row):
    history = row['history'].split() if pd.notna(row['history']) else []
    pos, neg = split_impressions(row['impressions'])

    return pd.Series({
        'history_ids': history,
        'positive_ids': pos,
        'negative_ids': neg
    })

In [ ]:
train_behavior_parsed = train_behaviors.copy()
dev_behavior_parsed = dev_behaviors.copy()

train_behavior_parsed[['history_ids', 'positive_ids', 'negative_ids']] = (
    train_behavior_parsed.apply(parse_behavior_row, axis=1)
)

dev_behavior_parsed[['history_ids', 'positive_ids', 'negative_ids']] = (
    dev_behavior_parsed.apply(parse_behavior_row, axis=1)
)

print("Parsed train and dev behaviors successfully.")

Parsed train and dev behaviors successfully.


In [ ]:
display(
    train_behavior_parsed[
        ['impression_id', 'user_id', 'history_ids', 'positive_ids', 'negative_ids']
    ].head()
)

,impression_id,user_id,history_ids,positive_ids,negative_ids
0,1,U13740,"[N55189, N42782, N34694, N45794, N18445, N6330...",[N55689],[N35729]
1,2,U91836,"[N31739, N6072, N63045, N23979, N35656, N43353...",[N17059],"[N20678, N39317, N58114, N20495, N42977, N2240..."
2,3,U73700,"[N10732, N25792, N7563, N21087, N41087, N5445,...",[N23814],"[N50014, N23877, N35389, N49712, N16844, N5968..."
3,4,U34670,"[N45729, N2203, N871, N53880, N41375, N43142, ...",[N49685],"[N35729, N33632, N27581]"
4,5,U8125,"[N10078, N56514, N14904, N33740]",[N8400],"[N39985, N36050, N16096, N22407, N60408, N6149..."


In [ ]:
sample_idx = 0

print("User ID:", train_behavior_parsed.loc[sample_idx, 'user_id'])
print("History IDs:", train_behavior_parsed.loc[sample_idx, 'history_ids'])
print("Positive clicked IDs:", train_behavior_parsed.loc[sample_idx, 'positive_ids'])
print("Negative non-clicked IDs:", train_behavior_parsed.loc[sample_idx, 'negative_ids'])

User ID: U13740
History IDs: ['N55189', 'N42782', 'N34694', 'N45794', 'N18445', 'N63302', 'N10414', 'N19347', 'N31801']
Positive clicked IDs: ['N55689']
Negative non-clicked IDs: ['N35729']


In [ ]:
train_behavior_parsed['history_len_ids'] = train_behavior_parsed['history_ids'].apply(len)
train_behavior_parsed['num_pos'] = train_behavior_parsed['positive_ids'].apply(len)
train_behavior_parsed['num_neg'] = train_behavior_parsed['negative_ids'].apply(len)

print("Average history length:", train_behavior_parsed['history_len_ids'].mean())
print("Average positives per impression:", train_behavior_parsed['num_pos'].mean())
print("Average negatives per impression:", train_behavior_parsed['num_neg'].mean())

Average history length: 32.53998662122129
Average positives per impression: 1.5057114643391838
Average negatives per impression: 35.72197623674067


In [ ]:
train_behavior_parsed.to_pickle(os.path.join(RESULTS_DIR, "train_behavior_parsed.pkl"))
dev_behavior_parsed.to_pickle(os.path.join(RESULTS_DIR, "dev_behavior_parsed.pkl"))

print("Saved train_behavior_parsed.pkl and dev_behavior_parsed.pkl")

Saved train_behavior_parsed.pkl and dev_behavior_parsed.pkl


## Step 5: Negative Sampling

For each positive click within an impression, 4 negative articles were randomly sampled from the non-clicked articles in the same impression. This produced training instances consisting of one positive candidate and four negative candidates, with labels formatted as `[1, 0, 0, 0, 0]`.

The user’s click history was also retrieved and truncated to the most recent 50 interactions. This step converts parsed behavior logs into model-ready training samples for neural recommendation, where the task becomes ranking the clicked article higher than the sampled negatives.

In [ ]:
import numpy as np

def create_negative_samples(behaviors_df, news_encoded, neg_k=4, max_history_len=50, seed=42):
    rng = np.random.default_rng(seed)
    samples = []

    for _, row in behaviors_df.iterrows():
        history_ids = row['history_ids'] if isinstance(row['history_ids'], list) else []
        positive_ids = row['positive_ids'] if isinstance(row['positive_ids'], list) else []
        negative_ids = row['negative_ids'] if isinstance(row['negative_ids'], list) else []

        # Encode user history (keep only valid news)
        history_encoded = [news_encoded[nid] for nid in history_ids if nid in news_encoded]
        history_encoded = history_encoded[-max_history_len:]

        # Skip if not enough data
        if len(positive_ids) == 0 or len(negative_ids) < neg_k:
            continue

        for pos_id in positive_ids:
            if pos_id not in news_encoded:
                continue

            # Sample exactly K negatives
            sampled_neg_ids = rng.choice(negative_ids, size=neg_k, replace=False)

            candidate_ids = [pos_id] + list(sampled_neg_ids)

            # Encode candidates
            candidate_encoded = [news_encoded[nid] for nid in candidate_ids if nid in news_encoded]

            # Labels: 1 positive + K negatives
            labels = [1] + [0] * neg_k

            # Ensure alignment
            if len(candidate_encoded) == len(labels):
                samples.append({
                    'history': history_encoded,
                    'candidate_ids': candidate_ids,
                    'candidates': candidate_encoded,
                    'labels': labels
                })

    return samples

In [ ]:
train_samples = create_negative_samples(
    behaviors_df=train_behavior_parsed,
    news_encoded=all_news_encoded,
    neg_k=4,
    max_history_len=50,
    seed=42
)

dev_samples = create_negative_samples(
    behaviors_df=dev_behavior_parsed,
    news_encoded=all_news_encoded,
    neg_k=4,
    max_history_len=50,
    seed=42
)

print("Train samples:", len(train_samples))
print("Dev samples:", len(dev_samples))

Train samples: 214962
Dev samples: 104697


In [ ]:
sample = train_samples[0]

print("History length:", len(sample['history']))
print("Candidate IDs:", sample['candidate_ids'])
print("Number of candidates:", len(sample['candidates']))
print("Labels:", sample['labels'])

History length: 50
Candidate IDs: ['N17059', np.str_('N42977'), np.str_('N20678'), np.str_('N22407'), np.str_('N14592')]
Number of candidates: 5
Labels: [1, 0, 0, 0, 0]


In [ ]:
print("One encoded candidate shape:", np.array(sample['candidates'][0]).shape)
print("Expected title length:", tokenizer.max_title_len)

One encoded candidate shape: (30,)
Expected title length: 30


In [ ]:
display(pd.DataFrame({
    'candidate_id': sample['candidate_ids'],
    'label': sample['labels']
}))

,candidate_id,label
0,N17059,1
1,N42977,0
2,N20678,0
3,N22407,0
4,N14592,0


In [ ]:
np.save(
    os.path.join(RESULTS_DIR, "train_samples.npy"),
    np.array(train_samples, dtype=object),
    allow_pickle=True
)

np.save(
    os.path.join(RESULTS_DIR, "dev_samples.npy"),
    np.array(dev_samples, dtype=object),
    allow_pickle=True
)

print("Saved train_samples.npy and dev_samples.npy")

Saved train_samples.npy and dev_samples.npy


## Step 6: Train/Validation Split

The provided MIND-small dataset includes predefined train and validation (dev) splits based on temporal ordering of user interactions.

In this project, these splits were used directly without any shuffling or mixing across the temporal boundary. All preprocessing steps, including tokenization, encoding, behavior parsing, and negative sampling, were performed separately on the training and validation sets. This ensures that the validation set represents future user interactions relative to the training data, maintaining a realistic evaluation setup for the recommendation model.

In [ ]:
#Quick Verfication
print("Train samples:", len(train_samples))
print("Dev samples:", len(dev_samples))

print("\nExample train user:", train_behavior_parsed.iloc[0]['user_id'])
print("Example dev user:", dev_behavior_parsed.iloc[0]['user_id'])

Train samples: 214962
Dev samples: 104697

Example train user: U13740
Example dev user: U80234
